# 01. CFLP instance 생성

Beasley(OR-Library) / Cornuejols-Sridharan-Thizy(1991) 계열의 CFLP instance generation 구조를 따라 4개의 instance를 생성한다.

- 위치: `U[0,1] x U[0,1]`
- 수요: `d_i ~ Normal(35, 5)` 를 반올림 후 최소 1로 clip
- 용량: `s_j ~ Uniform[10,160]` 을 `sum_j s_j = 1.5 * sum_i d_i` 가 되도록 정수 rescaling
- 고정비용: `f_j = U[0,90] + U[100,110] * sqrt(s_j)` (용량과 양의 상관)
- 운송 단가: `c_ij = 10 * dist_ij`

모든 instance는 설정 파일에 고정된 seed로 생성되므로 재현 가능하다. **SS와 MS는 동일한 instance data를 사용한다.**

In [ ]:
# 프로젝트 루트를 import 경로에 추가한다.
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.config import load_config, resolve_path

config = load_config(PROJECT_ROOT / "config" / "experiment_config.yaml")
DATA_DIR = resolve_path(config, "data_dir")
RAW_DIR = resolve_path(config, "raw_dir")
PROCESSED_DIR = resolve_path(config, "processed_dir")
FIGURE_DIR = resolve_path(config, "figure_dir")
SOLUTION_DIR = RAW_DIR / "solutions"
SOLUTION_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)
print("설정 로드 완료:", len(config["instances"]), "개 instance")


In [ ]:
from src.data_generator import generate_all_instances

instances = generate_all_instances(config)
for instance in instances:
    path = instance.save(DATA_DIR)
    print(f"{instance.name}: 저장 -> {path.name}")

## 생성된 instance 요약

전체 용량 / 전체 수요 비율이 목표값 1.5에 정확히 맞는지 확인한다.

In [ ]:
summary = pd.DataFrame(
    [
        {
            "instance": instance.name,
            "size": instance.num_customers,
            "seed": instance.seed,
            "total_demand": instance.total_demand,
            "total_capacity": instance.total_capacity,
            "capacity_ratio": round(instance.capacity_ratio, 4),
            "demand_min": int(instance.demands.min()),
            "demand_max": int(instance.demands.max()),
            "capacity_min": int(instance.capacities.min()),
            "capacity_max": int(instance.capacities.max()),
            "fixed_cost_mean": round(float(instance.fixed_costs.mean()), 2),
            "transport_cost_max": round(float(instance.transport_costs.max()), 4),
        }
        for instance in instances
    ]
)
summary.to_csv(RAW_DIR / "instance_summary.csv", index=False)
summary

## 용량-고정비용 상관관계 확인

고정비용을 모든 시설에 동일하게 주지 않았고, 용량과 양의 상관을 갖는지 수치로 확인한다.

In [ ]:
for instance in instances:
    correlation = float(
        np.corrcoef(instance.capacities, instance.fixed_costs)[0, 1]
    )
    print(f"{instance.name}: corr(s_j, f_j) = {correlation:+.4f}")

## 데이터 분포 시각화

위쪽 행은 고객(원, 크기 = 수요)과 시설(사각형, 크기 = 용량)의 위치이고, 아래쪽 행은 용량 대비 고정비용이다.

In [ ]:
from src.plotting import plot_instance_overview

path = plot_instance_overview(instances, FIGURE_DIR)
print("저장:", path)
from IPython.display import Image
Image(filename=str(path))